# 6. Hyperparameteroptimierung der klassischen Modellen mit Optuna

In diesem Notebook werden die aussichtsreichsten klassischen Modelle aus der Baseline-Phase mit Optuna optimiert. Die Optimierung erfolgt ausschließlich auf dem Trainingsdatensatz. Der Testdatensatz bleibt weiterhin unberührt und wird erst in der finalen Evaluation verwendet.

## 6.1. Ziel der Optimierung

Die bisherigen Experimente zeigen, dass das Szenario `Text + categorical` eine gute Balance zwischen Modellleistung und reduzierter Merkmalskomplexität liefert. Daher wird dieses Szenario für die Hyperparameteroptimierung als feste Datengrundlage verwendet.

Die Reihenfolge der Optimierung orientiert sich an der bisherigen Laufzeit: Zuerst wird `SGDClassifier(loss="hinge")` optimiert, da dieses Modell sehr schnell trainiert und dadurch mehr Suchläufe erlaubt. Anschließend wird `Logistic Regression balanced` optimiert, da dieses Modell in der Baseline den höchsten Macro-F1-Score erzielt hat.

Als Hauptmetrik wird weiterhin `Macro-F1` verwendet. Zusätzlich werden Standardabweichung, Train-Test-Gap, Weighted-F1, Balanced Accuracy und Laufzeit dokumentiert, damit die Modellauswahl nicht nur auf einem einzelnen Mittelwert basiert.

In [14]:
from pathlib import Path
import os
import re
import tempfile
import sys

import mlflow
import numpy as np
import optuna
import pandas as pd

from IPython.display import display
from joblib import Memory, dump
from tqdm.autonotebook import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

## 6.2. Daten laden

Es werden nur die Trainingsdaten geladen. Die Hyperparameteroptimierung darf den Testdatensatz nicht verwenden, damit die finale Modellbewertung unverzerrt bleibt.

In [15]:

# Hugging-Face-Konfiguration: Der Token wird nicht im Notebook gespeichert.
# Falls HF_TOKEN bereits als Umgebungsvariable oder Notebook-Variable existiert,
# wird er fuer Downloads vom Hugging Face Hub verwendet.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
hf_token = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN"))
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    hf_token = None

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import importlib
import model_evaluation
import model_evaluation.utils as model_evaluation_utils

importlib.reload(model_evaluation_utils)
importlib.reload(model_evaluation)

from model_evaluation import (
    combine_columns_as_text,
    display_model_evaluation_table,
    make_fasttext_label,
    safe_model_filename,
    ensure_val_metric_aliases,
    select_validation_result_columns,
)


sklearn_memory = Memory(
    location=project_root / "cache" / "sklearn",
    verbose=0,
)

model_output_dir = project_root / "models" / "optuna"
model_output_dir.mkdir(parents=True, exist_ok=True)


def save_fitted_model(model, model_name, X, y):
    """Fitten auf allen Trainingsdaten und als Joblib-Datei speichern."""
    model_path = model_output_dir / f"{safe_model_filename(model_name)}.joblib"
    model.fit(X, y)
    dump(model, model_path)
    return model_path

train_df = pd.read_csv(
    project_root / "data" / "processed" / "train_data.csv",
    sep=";",
    index_col=0,
    encoding="utf-8",
)

print("Trainingsdaten:", train_df.shape)
display(train_df.head())

Trainingsdaten: (45920, 28)


,name,geber,art,jahr,anschrift,politikbereich,zweck,betrag,empfaengerid,name_normalized,...,anschrift_standardised,zweck_standardised,empfaengerid_standardised,name_standardised_before_modelling,geber_standardised_before_modelling,art_standardised_before_modelling,anschrift_standardised_before_modelling,zweck_standardised_before_modelling,empfaengerid_standardised_before_modelling,betrag_standardised
id,,,,,,,,,,,,,,,,,,,,,
33655,Cashmere Radio e. V.,Senatsverwaltung für Kultur und Gesellschaftli...,Projektförderung,2024,"Frankfurter Allee 7, 10247 Berlin",Kultur,"signal2noise ? Art, Aesthetics And Social Prac...",120000,vr_035983,cashmere radio e. v.,...,"frankfurter allee 7, 10247 berlin-bezirk fried...","signal2noise ? art, aesthetics and social prac...",vr_035983,cashmere radio e. v.,senatsverwaltung für kultur und gesellschaftli...,projektförderung,"Frankfurter Allee 7, 10247 Berlin-Bezirk Fried...","signal2noise ? art, aesthetics and social prac...",vr_035983,11.695255
105910,Merantix Labs GmbH,"Senatsverwaltung für Wirtschaft, Energie und B...",Projektförderung,2022,"Max-Urich-Straße 3, 13355 Berlin",Wirtschaft,Errichtung einer Betriebsstätte,5481380,hrb_221397,merantix labs gmbh,...,"c/o ai campus, fachbereich bionik und evolutio...",errichtung einer betriebsstätte,hrb_221397,merantix labs gmbh,"senatsverwaltung für wirtschaft, energie und b...",projektförderung,"c/o AI Campus, Fachbereich Bionik und Evolutio...",errichtung einer betriebsstätte,hrb_221397,15.516868
67465,Georg-Kolbe-Stiftung,Senatsverwaltung für Kultur und Europa,Projektförderung,2020,"Sensburger Allee 25, 14055 Berlin",Kultur,Der absolute Tanz- Festival sculpture,70000,spr_100011,georg-kolbe-stiftung,...,"sensburger allee 25, 14055 berlin-bezirk charl...",der absolute tanz- festival sculpture,spr_100011,georg kolbe-stiftung,senatsverwaltung für kultur und europa,projektförderung,"Sensburger Allee 25, 14055 Berlin-Bezirk Charl...",der absolute tanz- festival sculpture,spr_100011,11.156265
159301,Verschiedene Gesellschaften bürgerlichen Rechts,"Senatsverwaltung für Inneres, Digitalisierung ...",Projektförderung,2023,'---',Sport,Kosten für die Beschäftigung von Übungsleitern,1380,NaN,verschiedene gesellschaften bürgerlichen rechts,...,deutschland,kosten für die beschäftigung von übungsleitern,NaN,verschiedene gesellschaften bürgerlichen rechts,"senatsverwaltung für inneres, digitalisierung ...",projektförderung,Deutschland,kosten für die beschäftigung von übungsleitern,NaN,7.230563
19495,Berliner Rugby-Club,Senatsverwaltung für Inneres und Sport,Projektförderung,2020,"Scharfestraße 12, 14169 Berlin",Sport,anteilige Finanzierung des Spielbetriebes der ...,9000,vr_002548,berliner rugby-club,...,"scharfestrasse 12, 14169 berlin-bezirk steglit...",anteilige finanzierung des spielbetriebes der ...,vr_002548,berliner rugby-club,senatsverwaltung für inneres und sport,projektförderung,"Scharfestraße 12, 14169 Berlin-Bezirk Steglitz...",anteilige finanzierung des spielbetriebes der ...,vr_002548,9.105091


## 6.3. Optimierungsszenario festlegen

Für das Tuning wird das Szenario `Text + categorical` verwendet. Numerische Merkmale werden in diesem Szenario bewusst nicht verwendet, da die Ablation darauf hindeutet, dass die Text- und kategorialen Merkmale für diese Aufgabe ausreichend stark sind.

In [16]:
target_column = "politikbereich"

text_features_optuna = [
    "name_standardised",
    "geber_standardised",
    "anschrift_standardised",
    "zweck_standardised",
]

categorical_features_optuna = [
    "art_standardised",
    "jahr",
]

numeric_features_optuna = []

feature_columns_optuna = (
    text_features_optuna
    + categorical_features_optuna
    + numeric_features_optuna
)

X_train = train_df[feature_columns_optuna].copy()
y_train = train_df[target_column].copy()

print("Verwendete Eingabespalten:")
display(pd.DataFrame({"Trainingsspalte": feature_columns_optuna}))

print("Anzahl Klassen:", y_train.nunique())

Verwendete Eingabespalten:


,Trainingsspalte
0,name_standardised
1,geber_standardised
2,anschrift_standardised
3,zweck_standardised
4,art_standardised
5,jahr


Anzahl Klassen: 32


## 6.4. Vorverarbeitung und Evaluationsrahmen

Die Vorverarbeitung entspricht der Baseline-Logik: Jede Textspalte wird separat mit TF-IDF verarbeitet, kategoriale Merkmale werden One-Hot-codiert. Für die Bewertung wird Stratified K-Fold Cross-Validation verwendet, damit die Klassenverteilung in den Folds möglichst stabil bleibt.

In [17]:
def flatten_column(values):
    """Konvertiert eine einzelne Spalte in ein eindimensionales String-Array."""
    return np.asarray(values, dtype=object).ravel()


text_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="",
            ),
        ),
        (
            "flatten",
            FunctionTransformer(
                flatten_column,
                validate=False,
            ),
        ),
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_df=0.90,
                ngram_range=(1, 2),
                sublinear_tf=True,
                max_features=150_000,
                dtype=np.float32,
            ),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)


def build_preprocessor(
    text_features,
    categorical_features,
):
    transformers = [
        (
            f"tfidf_{column}",
            text_transformer,
            [column],
        )
        for column in text_features
    ]

    if categorical_features:
        transformers.append(
            (
                "categorical",
                categorical_transformer,
                categorical_features,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


preprocessor_optuna = build_preprocessor(
    text_features_optuna,
    categorical_features_optuna,
)

n_splits = 4
cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42,
)

scoring = {
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
}

main_metric = "macro_f1"

## 6.5. Hilfsfunktionen für Optuna

Für jeden Trial wird eine Cross-Validation durchgeführt. Optuna maximiert den durchschnittlichen Test-Macro-F1-Score. Zusätzlich werden Train-Macro-F1, Generalization Gap, Standardabweichung und Laufzeit gespeichert, damit Overfitting und Stabilität beurteilt werden können.

In [18]:
def summarize_cv_results(cv_results):
    summary = {
        "fit_time_total_seconds": round(cv_results["fit_time"].sum(), 2),
        "fit_time_mean_seconds": round(cv_results["fit_time"].mean(), 2),
        "score_time_mean_seconds": round(cv_results["score_time"].mean(), 2),
    }

    for metric_name in scoring.keys():
        validation_scores = cv_results[f"test_{metric_name}"]
        summary[f"val_{metric_name}_mean"] = round(validation_scores.mean(), 4)
        summary[f"val_{metric_name}_std"] = round(validation_scores.std(), 4)
        summary[f"{metric_name}_mean"] = summary[f"val_{metric_name}_mean"]
        summary[f"{metric_name}_std"] = summary[f"val_{metric_name}_std"]

        train_key = f"train_{metric_name}"

        if train_key in cv_results:
            train_scores = cv_results[train_key]
            summary[f"train_{metric_name}_mean"] = round(train_scores.mean(), 4)
            summary[f"train_{metric_name}_std"] = round(train_scores.std(), 4)
            summary[f"generalization_gap_{metric_name}"] = round(
                summary[f"train_{metric_name}_mean"]
                - summary[f"val_{metric_name}_mean"],
                4,
            )

    return summary


def evaluate_model(model):
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True,
    )

    return summarize_cv_results(cv_results)


def log_optuna_trial(trial, model_name, summary):
    trial.set_user_attr("model_name", model_name)

    for metric_name, metric_value in summary.items():
        if pd.notna(metric_value):
            trial.set_user_attr(metric_name, metric_value)


def build_result_table(study, model_name):
    rows = []

    for trial in study.trials:
        if trial.value is None:
            continue

        row = {
            "Modell": model_name,
            "trial_number": trial.number,
            "objective_value": trial.value,
        }
        row.update(trial.params)
        row.update(trial.user_attrs)
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values("objective_value", ascending=False)
        .reset_index(drop=True)
    )

## 6.6. Optuna und MLflow vorbereiten

Die Optuna-Studien werden in einer SQLite-Datei gespeichert. Dadurch können bereits berechnete Trials später wiederverwendet werden. Zusätzlich werden die besten Ergebnisse als CSV-Dateien gespeichert und in MLflow dokumentiert.

In [19]:
optuna_storage_path = (
    project_root
    / "data"
    / "processed"
    / "optuna_studies.db"
)

optuna_storage_url = "sqlite:///" + optuna_storage_path.as_posix()

optuna_results_dir = project_root / "data" / "processed"
optuna_results_dir.mkdir(parents=True, exist_ok=True)

mlflow_tracking_uri = (project_root / "mlruns").as_uri()
mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("politikbereich_classifier_optuna")

sampler = optuna.samplers.TPESampler(seed=42)

print("Optuna storage:", optuna_storage_url)
print("MLflow tracking URI:", mlflow_tracking_uri)

Optuna storage: sqlite:///d:/toydev/schwarz-test/data/processed/optuna_studies.db
MLflow tracking URI: file:///d:/toydev/schwarz-test/mlruns


## 6.7. Studie 1: SGDClassifier mit Hinge Loss

Zuerst wird das schnelle SGD-SVM-Modell optimiert. Der Schwerpunkt liegt auf der Regularisierung (`alpha`), der Penalty-Struktur und der Konvergenztoleranz. `class_weight="balanced"` bleibt gesetzt, weil die Zielvariable deutlich unausgewogen ist.

In [20]:
def objective_sgd_hinge(trial):
    penalty = trial.suggest_categorical(
        "penalty",
        [
            "l2",
            "elasticnet",
        ],
    )

    classifier_params = {
        "loss": "hinge",
        "penalty": penalty,
        "alpha": trial.suggest_float(
            "alpha",
            1e-6,
            1e-3,
            log=True,
        ),
        "class_weight": "balanced",
        "max_iter": trial.suggest_categorical(
            "max_iter",
            [
                1000,
                2000,
                5000,
            ],
        ),
        "tol": trial.suggest_categorical(
            "tol",
            [
                1e-4,
                1e-3,
                1e-2,
            ],
        ),
        "average": trial.suggest_categorical(
            "average",
            [
                False,
                True,
            ],
        ),
        "random_state": 42,
        "n_jobs": -1,
    }

    if penalty == "elasticnet":
        classifier_params["l1_ratio"] = trial.suggest_float(
            "l1_ratio",
            0.05,
            0.50,
        )

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor_optuna,
            ),
            (
                "classifier",
                SGDClassifier(**classifier_params),
            ),
        ],
        memory=sklearn_memory,
    )

    summary = evaluate_model(model)
    log_optuna_trial(
        trial,
        "SGD hinge",
        summary,
    )

    return summary[f"val_{main_metric}_mean"]


sgd_hinge_study = optuna.create_study(
    study_name="sgd_hinge_text_categorical_macro_f1_cv4",
    direction="maximize",
    storage=optuna_storage_url,
    load_if_exists=True,
    sampler=sampler,
)

n_trials_sgd_hinge = 50

sgd_hinge_study.optimize(
    objective_sgd_hinge,
    n_trials=n_trials_sgd_hinge,
    show_progress_bar=True,
)

sgd_hinge_results = build_result_table(
    sgd_hinge_study,
    "SGD hinge",
)

display_model_evaluation_table(
    sgd_hinge_results.head(10),
    model_column_candidates=[
        "Modell",
    ],
    include_columns=[
        "trial_number",
        "objective_value",
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ],
    metric_prefixes=[
        "train_",
        "val_",
    ],
    sort_by="objective_value",
    ascending=False,
)

[I 2026-08-03 21:07:35,699] Using an existing study with name 'sgd_hinge_text_categorical_macro_f1_cv4' instead of creating a new one.


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-03 21:07:58,342] Trial 3 finished with value: 0.6747 and parameters: {'penalty': 'elasticnet', 'alpha': 0.000157029708840554, 'max_iter': 1000, 'tol': 0.001, 'average': False, 'l1_ratio': 0.48645943347289744}. Best is trial 1 with value: 0.8053.
[I 2026-08-03 21:08:19,801] Trial 4 finished with value: 0.8053 and parameters: {'penalty': 'l2', 'alpha': 3.5113563139704077e-06, 'max_iter': 5000, 'tol': 0.01, 'average': True}. Best is trial 1 with value: 0.8053.
[I 2026-08-03 21:08:46,124] Trial 5 finished with value: 0.0719 and parameters: {'penalty': 'elasticnet', 'alpha': 0.0002267398652378039, 'max_iter': 5000, 'tol': 0.001, 'average': True, 'l1_ratio': 0.4845344148835517}. Best is trial 1 with value: 0.8053.
[I 2026-08-03 21:09:10,121] Trial 6 finished with value: 0.8079 and parameters: {'penalty': 'l2', 'alpha': 1.9634341572933354e-06, 'max_iter': 1000, 'tol': 0.01, 'average': True}. Best is trial 6 with value: 0.8079.
[I 2026-08-03 21:09:40,618] Trial 7 finished with value

,Modell,trial_number,objective_value,penalty,alpha,l1_ratio,max_iter,tol,average,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,SGD hinge,27,0.8523,l2,0.000012,NaN,2000,0.0001,False,0.9924,0.0002,0.9917,0.0011,0.9747,0.0048,0.9925,0.0002,0.9413,0.0025,0.8554,0.0029,0.8523,0.0029,0.9414,0.0023,12.08
1,SGD hinge,34,0.8517,l2,0.000021,NaN,5000,0.0001,False,0.9900,0.0005,0.9879,0.0047,0.9658,0.0072,0.9901,0.0005,0.9406,0.0022,0.8552,0.0085,0.8517,0.0062,0.9409,0.0019,10.87
2,SGD hinge,47,0.8515,l2,0.000016,NaN,5000,0.0001,False,0.9910,0.0003,0.9896,0.0034,0.9680,0.0106,0.9912,0.0003,0.9414,0.0018,0.8556,0.0064,0.8515,0.0058,0.9416,0.0018,12.59
3,SGD hinge,29,0.8509,l2,0.000019,NaN,5000,0.0001,False,0.9904,0.0004,0.9884,0.0047,0.9669,0.0076,0.9906,0.0003,0.9405,0.0028,0.8520,0.0115,0.8509,0.0086,0.9408,0.0026,10.83
4,SGD hinge,38,0.8500,l2,0.000020,NaN,5000,0.0001,False,0.9899,0.0004,0.9880,0.0045,0.9646,0.0056,0.9900,0.0004,0.9409,0.0016,0.8542,0.0127,0.8500,0.0055,0.9413,0.0015,11.33
5,SGD hinge,48,0.8499,l2,0.000006,NaN,5000,0.0001,False,0.9934,0.0003,0.9929,0.0016,0.9748,0.0069,0.9935,0.0003,0.9397,0.0026,0.8527,0.0039,0.8499,0.0025,0.9398,0.0023,13.57
6,SGD hinge,42,0.8498,l2,0.000017,NaN,5000,0.0001,False,0.9910,0.0002,0.9887,0.0043,0.9652,0.0049,0.9911,0.0003,0.9407,0.0019,0.8529,0.0131,0.8498,0.0069,0.9409,0.0018,16.20
7,SGD hinge,52,0.8496,l2,0.000005,NaN,5000,0.0001,False,0.9938,0.0002,0.9930,0.0014,0.9763,0.0050,0.9938,0.0002,0.9397,0.0037,0.8473,0.0044,0.8496,0.0057,0.9398,0.0036,14.69
8,SGD hinge,43,0.8488,l2,0.000020,NaN,5000,0.0001,False,0.9903,0.0001,0.9876,0.0048,0.9644,0.0064,0.9904,0.0002,0.9403,0.0019,0.8521,0.0143,0.8488,0.0112,0.9406,0.0019,12.49
9,SGD hinge,33,0.8486,l2,0.000019,NaN,5000,0.0001,False,0.9906,0.0004,0.9882,0.0047,0.9656,0.0077,0.9908,0.0004,0.9405,0.0030,0.8537,0.0113,0.8486,0.0068,0.9408,0.0026,11.21


,Modell,trial_number,objective_value,penalty,alpha,l1_ratio,max_iter,tol,average,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,SGD hinge,27,0.8523,l2,0.000012,NaN,2000,0.0001,False,0.9924,0.0002,0.9917,0.0011,0.9747,0.0048,0.9925,0.0002,0.9413,0.0025,0.8554,0.0029,0.8523,0.0029,0.9414,0.0023,12.08
1,SGD hinge,34,0.8517,l2,0.000021,NaN,5000,0.0001,False,0.9900,0.0005,0.9879,0.0047,0.9658,0.0072,0.9901,0.0005,0.9406,0.0022,0.8552,0.0085,0.8517,0.0062,0.9409,0.0019,10.87
2,SGD hinge,47,0.8515,l2,0.000016,NaN,5000,0.0001,False,0.9910,0.0003,0.9896,0.0034,0.9680,0.0106,0.9912,0.0003,0.9414,0.0018,0.8556,0.0064,0.8515,0.0058,0.9416,0.0018,12.59
3,SGD hinge,29,0.8509,l2,0.000019,NaN,5000,0.0001,False,0.9904,0.0004,0.9884,0.0047,0.9669,0.0076,0.9906,0.0003,0.9405,0.0028,0.8520,0.0115,0.8509,0.0086,0.9408,0.0026,10.83
4,SGD hinge,38,0.8500,l2,0.000020,NaN,5000,0.0001,False,0.9899,0.0004,0.9880,0.0045,0.9646,0.0056,0.9900,0.0004,0.9409,0.0016,0.8542,0.0127,0.8500,0.0055,0.9413,0.0015,11.33
5,SGD hinge,48,0.8499,l2,0.000006,NaN,5000,0.0001,False,0.9934,0.0003,0.9929,0.0016,0.9748,0.0069,0.9935,0.0003,0.9397,0.0026,0.8527,0.0039,0.8499,0.0025,0.9398,0.0023,13.57
6,SGD hinge,42,0.8498,l2,0.000017,NaN,5000,0.0001,False,0.9910,0.0002,0.9887,0.0043,0.9652,0.0049,0.9911,0.0003,0.9407,0.0019,0.8529,0.0131,0.8498,0.0069,0.9409,0.0018,16.20
7,SGD hinge,52,0.8496,l2,0.000005,NaN,5000,0.0001,False,0.9938,0.0002,0.9930,0.0014,0.9763,0.0050,0.9938,0.0002,0.9397,0.0037,0.8473,0.0044,0.8496,0.0057,0.9398,0.0036,14.69
8,SGD hinge,43,0.8488,l2,0.000020,NaN,5000,0.0001,False,0.9903,0.0001,0.9876,0.0048,0.9644,0.0064,0.9904,0.0002,0.9403,0.0019,0.8521,0.0143,0.8488,0.0112,0.9406,0.0019,12.49
9,SGD hinge,33,0.8486,l2,0.000019,NaN,5000,0.0001,False,0.9906,0.0004,0.9882,0.0047,0.9656,0.0077,0.9908,0.0004,0.9405,0.0030,0.8537,0.0113,0.8486,0.0068,0.9408,0.0026,11.21


## 6.8. Studie 2: Linear SVC unweighted

Als zweites Modell wird die ungewichtete `LinearSVC`-Variante optimiert. Dieses Modell war in der Baseline ein starker Einzelkandidat nach Macro-F1 und wird deshalb gezielt weiter untersucht. Optimiert wird vor allem die Regularisierung `C`. Die Klassengewichtung bleibt bewusst deaktiviert, damit diese Studie die präzisere, weniger aggressive Variante des SVM-Modells abbildet.

In [21]:
def objective_linear_svc_unweighted(trial):
    classifier_params = {
        "class_weight": None,
        "C": trial.suggest_float(
            "C",
            0.03,
            0.50,
            log=True,
        ),
        "max_iter": trial.suggest_categorical(
            "max_iter",
            [
                10000,
                20000,
                30000,
            ],
        ),
        "tol": trial.suggest_categorical(
            "tol",
            [
                1e-4,
                5e-4,
                1e-3,
            ],
        ),
        "dual": "auto",
        "random_state": 42,
    }

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor_optuna,
            ),
            (
                "classifier",
                LinearSVC(**classifier_params),
            ),
        ],
        memory=sklearn_memory,
    )

    summary = evaluate_model(model)
    log_optuna_trial(
        trial,
        "Linear SVC unweighted",
        summary,
    )

    return summary[f"val_{main_metric}_mean"]


linear_svc_unweighted_study = optuna.create_study(
    study_name="linear_svc_unweighted_macro_f1_cv4",
    direction="maximize",
    storage=optuna_storage_url,
    load_if_exists=True,
    sampler=sampler,
)

n_trials_linear_svc_unweighted = 20

linear_svc_unweighted_study.optimize(
    objective_linear_svc_unweighted,
    n_trials=n_trials_linear_svc_unweighted,
    show_progress_bar=True,
)

linear_svc_unweighted_results = build_result_table(
    linear_svc_unweighted_study,
    "Linear SVC unweighted",
)

display_model_evaluation_table(
    linear_svc_unweighted_results.head(10),
    model_column_candidates=[
        "Modell",
    ],
    include_columns=[
        "trial_number",
        "objective_value",
        "C",
        "class_weight",
        "max_iter",
        "tol",
    ],
    metric_prefixes=[
        "train_",
        "val_",
    ],
    sort_by="objective_value",
    ascending=False,
)

[I 2026-08-03 21:30:41,550] A new study created in RDB with name: linear_svc_unweighted_macro_f1_cv4


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-03 21:31:21,248] Trial 0 finished with value: 0.849 and parameters: {'C': 0.25508534886550255, 'max_iter': 20000, 'tol': 0.0001}. Best is trial 0 with value: 0.849.
[I 2026-08-03 21:31:55,240] Trial 1 finished with value: 0.7723 and parameters: {'C': 0.04063949100106302, 'max_iter': 20000, 'tol': 0.0005}. Best is trial 0 with value: 0.849.
[I 2026-08-03 21:32:24,945] Trial 2 finished with value: 0.8207 and parameters: {'C': 0.09518031025046533, 'max_iter': 10000, 'tol': 0.001}. Best is trial 0 with value: 0.849.
[I 2026-08-03 21:32:55,461] Trial 3 finished with value: 0.852 and parameters: {'C': 0.29142186287192495, 'max_iter': 20000, 'tol': 0.0005}. Best is trial 3 with value: 0.852.
[I 2026-08-03 21:33:26,298] Trial 4 finished with value: 0.852 and parameters: {'C': 0.2908646874193105, 'max_iter': 10000, 'tol': 0.001}. Best is trial 3 with value: 0.852.
[I 2026-08-03 21:33:55,337] Trial 5 finished with value: 0.8536 and parameters: {'C': 0.3379121764859049, 'max_iter': 200

,Modell,trial_number,objective_value,C,max_iter,tol,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,Linear SVC unweighted,6,0.8575,0.425808,30000,0.0005,0.9931,0.0002,0.9829,0.0012,0.9864,0.0007,0.9931,0.0002,0.9425,0.0011,0.8415,0.0211,0.8575,0.0142,0.9418,0.0011,16.13
1,Linear SVC unweighted,12,0.8571,0.489375,30000,0.0010,0.9936,0.0002,0.9842,0.0011,0.9872,0.0005,0.9936,0.0002,0.9431,0.0009,0.8415,0.0191,0.8571,0.0134,0.9425,0.0008,16.79
2,Linear SVC unweighted,11,0.8563,0.475700,30000,0.0010,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,15.19
3,Linear SVC unweighted,13,0.8563,0.475852,30000,0.0005,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,18.10
4,Linear SVC unweighted,18,0.8563,0.475115,30000,0.0010,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,20.84
5,Linear SVC unweighted,17,0.8550,0.372417,30000,0.0005,0.9924,0.0002,0.9813,0.0012,0.9854,0.0007,0.9924,0.0002,0.9421,0.0009,0.8399,0.0210,0.8550,0.0165,0.9414,0.0008,19.33
6,Linear SVC unweighted,5,0.8536,0.337912,20000,0.0010,0.9917,0.0001,0.9797,0.0013,0.9843,0.0008,0.9917,0.0001,0.9416,0.0008,0.8379,0.0208,0.8536,0.0164,0.9408,0.0008,15.98
7,Linear SVC unweighted,3,0.8520,0.291422,20000,0.0005,0.9903,0.0002,0.9757,0.0018,0.9818,0.0009,0.9903,0.0002,0.9406,0.0008,0.8346,0.0189,0.8520,0.0159,0.9398,0.0007,16.62
8,Linear SVC unweighted,4,0.8520,0.290865,10000,0.0010,0.9903,0.0002,0.9757,0.0018,0.9817,0.0009,0.9903,0.0002,0.9406,0.0008,0.8346,0.0189,0.8520,0.0159,0.9398,0.0007,16.24
9,Linear SVC unweighted,0,0.8490,0.255085,20000,0.0001,0.9888,0.0003,0.9704,0.0022,0.9784,0.0013,0.9888,0.0003,0.9393,0.0011,0.8310,0.0191,0.8490,0.0164,0.9384,0.0011,21.45


,Modell,trial_number,objective_value,C,max_iter,tol,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,Linear SVC unweighted,6,0.8575,0.425808,30000,0.0005,0.9931,0.0002,0.9829,0.0012,0.9864,0.0007,0.9931,0.0002,0.9425,0.0011,0.8415,0.0211,0.8575,0.0142,0.9418,0.0011,16.13
1,Linear SVC unweighted,12,0.8571,0.489375,30000,0.0010,0.9936,0.0002,0.9842,0.0011,0.9872,0.0005,0.9936,0.0002,0.9431,0.0009,0.8415,0.0191,0.8571,0.0134,0.9425,0.0008,16.79
2,Linear SVC unweighted,11,0.8563,0.475700,30000,0.0010,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,15.19
3,Linear SVC unweighted,13,0.8563,0.475852,30000,0.0005,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,18.10
4,Linear SVC unweighted,18,0.8563,0.475115,30000,0.0010,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,20.84
5,Linear SVC unweighted,17,0.8550,0.372417,30000,0.0005,0.9924,0.0002,0.9813,0.0012,0.9854,0.0007,0.9924,0.0002,0.9421,0.0009,0.8399,0.0210,0.8550,0.0165,0.9414,0.0008,19.33
6,Linear SVC unweighted,5,0.8536,0.337912,20000,0.0010,0.9917,0.0001,0.9797,0.0013,0.9843,0.0008,0.9917,0.0001,0.9416,0.0008,0.8379,0.0208,0.8536,0.0164,0.9408,0.0008,15.98
7,Linear SVC unweighted,3,0.8520,0.291422,20000,0.0005,0.9903,0.0002,0.9757,0.0018,0.9818,0.0009,0.9903,0.0002,0.9406,0.0008,0.8346,0.0189,0.8520,0.0159,0.9398,0.0007,16.62
8,Linear SVC unweighted,4,0.8520,0.290865,10000,0.0010,0.9903,0.0002,0.9757,0.0018,0.9817,0.0009,0.9903,0.0002,0.9406,0.0008,0.8346,0.0189,0.8520,0.0159,0.9398,0.0007,16.24
9,Linear SVC unweighted,0,0.8490,0.255085,20000,0.0001,0.9888,0.0003,0.9704,0.0022,0.9784,0.0013,0.9888,0.0003,0.9393,0.0011,0.8310,0.0191,0.8490,0.0164,0.9384,0.0011,21.45


## 6.9. Studie 3: Linear SVC balanced

Die dritte Studie optimiert die gewichtete `LinearSVC`-Variante. Diese Variante ist methodisch relevant, weil `class_weight="balanced"` kleine Klassen während des Trainings stärker berücksichtigt. Dadurch lässt sich prüfen, ob die Rare-Class-Erkennung gegenüber der ungewichteten SVC verbessert werden kann.

In [22]:
def objective_linear_svc_balanced(trial):
    classifier_params = {
        "class_weight": "balanced",
        "C": trial.suggest_float(
            "C",
            0.01,
            0.30,
            log=True,
        ),
        "max_iter": trial.suggest_categorical(
            "max_iter",
            [
                10000,
                20000,
                30000,
            ],
        ),
        "tol": trial.suggest_categorical(
            "tol",
            [
                1e-4,
                5e-4,
                1e-3,
            ],
        ),
        "dual": "auto",
        "random_state": 42,
    }

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor_optuna,
            ),
            (
                "classifier",
                LinearSVC(**classifier_params),
            ),
        ],
        memory=sklearn_memory,
    )

    summary = evaluate_model(model)
    log_optuna_trial(
        trial,
        "Linear SVC balanced",
        summary,
    )

    return summary[f"val_{main_metric}_mean"]


linear_svc_balanced_study = optuna.create_study(
    study_name="linear_svc_balanced_macro_f1_cv4",
    direction="maximize",
    storage=optuna_storage_url,
    load_if_exists=True,
    sampler=sampler,
)

n_trials_linear_svc_balanced = 20

linear_svc_balanced_study.optimize(
    objective_linear_svc_balanced,
    n_trials=n_trials_linear_svc_balanced,
    show_progress_bar=True,
)

linear_svc_balanced_results = build_result_table(
    linear_svc_balanced_study,
    "Linear SVC balanced",
)

display_model_evaluation_table(
    linear_svc_balanced_results.head(10),
    model_column_candidates=[
        "Modell",
    ],
    include_columns=[
        "trial_number",
        "objective_value",
        "C",
        "class_weight",
        "max_iter",
        "tol",
    ],
    metric_prefixes=[
        "train_",
        "val_",
    ],
    sort_by="objective_value",
    ascending=False,
)

[I 2026-08-03 21:41:02,667] A new study created in RDB with name: linear_svc_balanced_macro_f1_cv4


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-03 21:41:33,576] Trial 0 finished with value: 0.8361 and parameters: {'C': 0.0862581622692092, 'max_iter': 30000, 'tol': 0.0001}. Best is trial 0 with value: 0.8361.
[I 2026-08-03 21:42:08,803] Trial 1 finished with value: 0.8327 and parameters: {'C': 0.07461403484816762, 'max_iter': 10000, 'tol': 0.0005}. Best is trial 0 with value: 0.8361.
[I 2026-08-03 21:42:44,855] Trial 2 finished with value: 0.8398 and parameters: {'C': 0.10485733570643786, 'max_iter': 20000, 'tol': 0.001}. Best is trial 2 with value: 0.8398.
[I 2026-08-03 21:43:13,639] Trial 3 finished with value: 0.8462 and parameters: {'C': 0.1976681084946931, 'max_iter': 30000, 'tol': 0.0001}. Best is trial 3 with value: 0.8462.
[I 2026-08-03 21:43:43,251] Trial 4 finished with value: 0.7437 and parameters: {'C': 0.01372537052078299, 'max_iter': 20000, 'tol': 0.001}. Best is trial 3 with value: 0.8462.
[I 2026-08-03 21:44:10,837] Trial 5 finished with value: 0.8477 and parameters: {'C': 0.211417353855613, 'max_iter

,Modell,trial_number,objective_value,C,max_iter,tol,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,Linear SVC balanced,15,0.8516,0.299297,10000,0.0001,0.9878,0.0003,0.9918,0.0003,0.9811,0.0009,0.9879,0.0003,0.9391,0.0017,0.8593,0.0123,0.8516,0.0102,0.9395,0.0015,19.74
1,Linear SVC balanced,19,0.8516,0.298967,10000,0.0001,0.9878,0.0003,0.9918,0.0003,0.9811,0.0009,0.9879,0.0003,0.9391,0.0016,0.8593,0.0123,0.8516,0.0102,0.9395,0.0015,16.14
2,Linear SVC balanced,11,0.8516,0.293942,10000,0.0010,0.9876,0.0003,0.9917,0.0003,0.9807,0.0008,0.9877,0.0003,0.9390,0.0016,0.8594,0.0122,0.8516,0.0099,0.9395,0.0014,16.65
3,Linear SVC balanced,10,0.8510,0.284531,10000,0.0010,0.9873,0.0003,0.9915,0.0003,0.9804,0.0009,0.9874,0.0002,0.9387,0.0015,0.8594,0.0123,0.8510,0.0095,0.9392,0.0013,14.91
4,Linear SVC balanced,12,0.8510,0.281263,10000,0.0010,0.9872,0.0003,0.9915,0.0003,0.9803,0.0009,0.9873,0.0002,0.9387,0.0014,0.8594,0.0123,0.8510,0.0095,0.9391,0.0012,16.52
5,Linear SVC balanced,14,0.8479,0.166376,10000,0.0010,0.9810,0.0006,0.9871,0.0010,0.9705,0.0009,0.9811,0.0006,0.9337,0.0008,0.8587,0.0102,0.8479,0.0061,0.9343,0.0006,14.52
6,Linear SVC balanced,5,0.8477,0.211417,10000,0.0010,0.9840,0.0004,0.9891,0.0008,0.9765,0.0011,0.9841,0.0004,0.9361,0.0008,0.8575,0.0128,0.8477,0.0098,0.9367,0.0006,15.94
7,Linear SVC balanced,3,0.8462,0.197668,30000,0.0001,0.9833,0.0003,0.9886,0.0008,0.9759,0.0010,0.9835,0.0003,0.9355,0.0010,0.8571,0.0123,0.8462,0.0089,0.9361,0.0008,16.69
8,Linear SVC balanced,16,0.8462,0.150803,20000,0.0001,0.9796,0.0008,0.9861,0.0010,0.9674,0.0011,0.9797,0.0007,0.9322,0.0010,0.8576,0.0100,0.8462,0.0057,0.9328,0.0007,17.04
9,Linear SVC balanced,18,0.8446,0.138620,10000,0.0001,0.9783,0.0006,0.9852,0.0008,0.9656,0.0011,0.9785,0.0006,0.9311,0.0009,0.8572,0.0099,0.8446,0.0054,0.9317,0.0007,14.86


,Modell,trial_number,objective_value,C,max_iter,tol,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,Linear SVC balanced,15,0.8516,0.299297,10000,0.0001,0.9878,0.0003,0.9918,0.0003,0.9811,0.0009,0.9879,0.0003,0.9391,0.0017,0.8593,0.0123,0.8516,0.0102,0.9395,0.0015,19.74
1,Linear SVC balanced,19,0.8516,0.298967,10000,0.0001,0.9878,0.0003,0.9918,0.0003,0.9811,0.0009,0.9879,0.0003,0.9391,0.0016,0.8593,0.0123,0.8516,0.0102,0.9395,0.0015,16.14
2,Linear SVC balanced,11,0.8516,0.293942,10000,0.0010,0.9876,0.0003,0.9917,0.0003,0.9807,0.0008,0.9877,0.0003,0.9390,0.0016,0.8594,0.0122,0.8516,0.0099,0.9395,0.0014,16.65
3,Linear SVC balanced,10,0.8510,0.284531,10000,0.0010,0.9873,0.0003,0.9915,0.0003,0.9804,0.0009,0.9874,0.0002,0.9387,0.0015,0.8594,0.0123,0.8510,0.0095,0.9392,0.0013,14.91
4,Linear SVC balanced,12,0.8510,0.281263,10000,0.0010,0.9872,0.0003,0.9915,0.0003,0.9803,0.0009,0.9873,0.0002,0.9387,0.0014,0.8594,0.0123,0.8510,0.0095,0.9391,0.0012,16.52
5,Linear SVC balanced,14,0.8479,0.166376,10000,0.0010,0.9810,0.0006,0.9871,0.0010,0.9705,0.0009,0.9811,0.0006,0.9337,0.0008,0.8587,0.0102,0.8479,0.0061,0.9343,0.0006,14.52
6,Linear SVC balanced,5,0.8477,0.211417,10000,0.0010,0.9840,0.0004,0.9891,0.0008,0.9765,0.0011,0.9841,0.0004,0.9361,0.0008,0.8575,0.0128,0.8477,0.0098,0.9367,0.0006,15.94
7,Linear SVC balanced,3,0.8462,0.197668,30000,0.0001,0.9833,0.0003,0.9886,0.0008,0.9759,0.0010,0.9835,0.0003,0.9355,0.0010,0.8571,0.0123,0.8462,0.0089,0.9361,0.0008,16.69
8,Linear SVC balanced,16,0.8462,0.150803,20000,0.0001,0.9796,0.0008,0.9861,0.0010,0.9674,0.0011,0.9797,0.0007,0.9322,0.0010,0.8576,0.0100,0.8462,0.0057,0.9328,0.0007,17.04
9,Linear SVC balanced,18,0.8446,0.138620,10000,0.0001,0.9783,0.0006,0.9852,0.0008,0.9656,0.0011,0.9785,0.0006,0.9311,0.0009,0.8572,0.0099,0.8446,0.0054,0.9317,0.0007,14.86


## 6.10. Ergebnisse der optimierten Modellkandidaten vergleichen und speichern

Zum Abschluss werden die besten Trials der drei optimierten Modellkandidaten zusammengeführt: `SGD hinge`, `Linear SVC unweighted` und `Linear SVC balanced`. Dadurch wird sichtbar, welche Hyperparameterkombinationen auf Basis der Cross-Validation am stärksten abschneiden.

In [23]:
model_tuning_results = pd.concat(
    [
        sgd_hinge_results,
        linear_svc_unweighted_results,
        linear_svc_balanced_results,
    ],
    ignore_index=True,
)

model_tuning_results = (
    model_tuning_results
    .sort_values(
        "objective_value",
        ascending=False,
    )
    .reset_index(drop=True)
)

optuna_results = model_tuning_results.copy()

display_model_evaluation_table(
    model_tuning_results.head(10),
    model_column_candidates=[
        "Modell",
    ],
    include_columns=[
        "trial_number",
        "objective_value",
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ],
    metric_prefixes=[
        "train_",
        "val_",
    ],
    sort_by="objective_value",
    ascending=False,
)

optuna_results_path = (
    optuna_results_dir
    / "optuna_tuning_results.csv"
)

sgd_hinge_results_path = (
    optuna_results_dir
    / "optuna_sgd_hinge_results.csv"
)

linear_svc_unweighted_results_path = (
    optuna_results_dir
    / "optuna_linear_svc_unweighted_results.csv"
)

linear_svc_balanced_results_path = (
    optuna_results_dir
    / "optuna_linear_svc_balanced_results.csv"
)

model_tuning_results_path = (
    optuna_results_dir
    / "optuna_model_tuning_results.csv"
)

optuna_results.to_csv(
    optuna_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

sgd_hinge_results.to_csv(
    sgd_hinge_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

linear_svc_unweighted_results.to_csv(
    linear_svc_unweighted_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

linear_svc_balanced_results.to_csv(
    linear_svc_balanced_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

model_tuning_results.to_csv(
    model_tuning_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

print("Gespeichert:", optuna_results_path)
print("Gespeichert:", sgd_hinge_results_path)
print("Gespeichert:", linear_svc_unweighted_results_path)
print("Gespeichert:", linear_svc_balanced_results_path)
print("Gespeichert:", model_tuning_results_path)

,Modell,trial_number,objective_value,penalty,alpha,l1_ratio,C,max_iter,tol,average,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,Linear SVC unweighted,6,0.8575,NaN,NaN,NaN,0.425808,30000,0.0005,NaN,0.9931,0.0002,0.9829,0.0012,0.9864,0.0007,0.9931,0.0002,0.9425,0.0011,0.8415,0.0211,0.8575,0.0142,0.9418,0.0011,16.13
1,Linear SVC unweighted,12,0.8571,NaN,NaN,NaN,0.489375,30000,0.0010,NaN,0.9936,0.0002,0.9842,0.0011,0.9872,0.0005,0.9936,0.0002,0.9431,0.0009,0.8415,0.0191,0.8571,0.0134,0.9425,0.0008,16.79
2,Linear SVC unweighted,13,0.8563,NaN,NaN,NaN,0.475852,30000,0.0005,NaN,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,18.10
3,Linear SVC unweighted,11,0.8563,NaN,NaN,NaN,0.475700,30000,0.0010,NaN,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,15.19
4,Linear SVC unweighted,18,0.8563,NaN,NaN,NaN,0.475115,30000,0.0010,NaN,0.9935,0.0002,0.9838,0.0013,0.9870,0.0007,0.9935,0.0002,0.9429,0.0007,0.8405,0.0191,0.8563,0.0133,0.9422,0.0007,20.84
5,Linear SVC unweighted,17,0.8550,NaN,NaN,NaN,0.372417,30000,0.0005,NaN,0.9924,0.0002,0.9813,0.0012,0.9854,0.0007,0.9924,0.0002,0.9421,0.0009,0.8399,0.0210,0.8550,0.0165,0.9414,0.0008,19.33
6,Linear SVC unweighted,5,0.8536,NaN,NaN,NaN,0.337912,20000,0.0010,NaN,0.9917,0.0001,0.9797,0.0013,0.9843,0.0008,0.9917,0.0001,0.9416,0.0008,0.8379,0.0208,0.8536,0.0164,0.9408,0.0008,15.98
7,SGD hinge,27,0.8523,l2,0.000012,NaN,NaN,2000,0.0001,False,0.9924,0.0002,0.9917,0.0011,0.9747,0.0048,0.9925,0.0002,0.9413,0.0025,0.8554,0.0029,0.8523,0.0029,0.9414,0.0023,12.08
8,Linear SVC unweighted,3,0.8520,NaN,NaN,NaN,0.291422,20000,0.0005,NaN,0.9903,0.0002,0.9757,0.0018,0.9818,0.0009,0.9903,0.0002,0.9406,0.0008,0.8346,0.0189,0.8520,0.0159,0.9398,0.0007,16.62
9,Linear SVC unweighted,4,0.8520,NaN,NaN,NaN,0.290865,10000,0.0010,NaN,0.9903,0.0002,0.9757,0.0018,0.9817,0.0009,0.9903,0.0002,0.9406,0.0008,0.8346,0.0189,0.8520,0.0159,0.9398,0.0007,16.24


Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_tuning_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_sgd_hinge_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_linear_svc_unweighted_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_linear_svc_balanced_results.csv
Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_model_tuning_results.csv


Die Optuna-Ergebnisse zeigen, ob sich die ausgewählten Modellkandidaten durch gezielte Hyperparameterwahl weiter verbessern lassen. Die Werte stammen aus Cross-Validation auf den Trainingsdaten und dienen ausschließlich der Modellauswahl. Die finale Aussage zur Generalisierung erfolgt erst im separaten Testdatensatz.

## 6.11. Bestes Einzelmodell aus der Tuning-Phase speichern

Der beste Trial nach Macro-F1 wird als zusätzlicher Modellkandidat gespeichert. Die vollständigen Ergebnisdateien der drei Studien bleiben ebenfalls erhalten, damit die Tuning-Resultate später nachvollziehbar verglichen werden können.

In [24]:
best_trial_summary = optuna_results.iloc[0]

print("Bestes Modell:", best_trial_summary["Modell"])
print("Trial:", best_trial_summary["trial_number"])
print("Macro-F1:", best_trial_summary["val_macro_f1_mean"])

if "generalization_gap_macro_f1" in best_trial_summary:
    print(
        "Generalization Gap Macro-F1:",
        best_trial_summary["generalization_gap_macro_f1"],
    )

best_trial_display = display_model_evaluation_table(
    optuna_results.head(1),
    model_column_candidates=[
        "Modell",
    ],
    include_columns=[
        "trial_number",
        "objective_value",
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ],
    metric_prefixes=[
        "train_",
        "val_",
    ],
)

display(
    best_trial_display.iloc[0].to_frame("Wert")
)

best_params = {
    key: value
    for key, value in best_trial_summary.items()
    if key in [
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ]
    and pd.notna(value)
}

if best_trial_summary["Modell"] == "SGD hinge":
    classifier_params = {
        "loss": "hinge",
        "penalty": best_params.get("penalty", "l2"),
        "alpha": float(best_params.get("alpha", 1e-4)),
        "class_weight": "balanced",
        "max_iter": int(best_params.get("max_iter", 1000)),
        "tol": float(best_params.get("tol", 1e-3)),
        "average": bool(best_params.get("average", False)),
        "random_state": 42,
        "n_jobs": -1,
    }

    if classifier_params["penalty"] == "elasticnet":
        classifier_params["l1_ratio"] = float(
            best_params.get("l1_ratio", 0.15)
        )

    final_classifier = SGDClassifier(**classifier_params)

elif best_trial_summary["Modell"] in [
    "Linear SVC unweighted",
    "Linear SVC balanced",
]:
    class_weight = (
        None
        if best_trial_summary["Modell"] == "Linear SVC unweighted"
        else "balanced"
    )

    final_classifier = LinearSVC(
        class_weight=class_weight,
        C=float(best_params.get("C", 0.1)),
        max_iter=int(best_params.get("max_iter", 20000)),
        tol=float(best_params.get("tol", 1e-3)),
        dual="auto",
        random_state=42,
    )

else:
    raise ValueError(
        f"Unbekanntes Modell: {best_trial_summary['Modell']}"
    )

best_model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_optuna,
        ),
        (
            "classifier",
            final_classifier,
        ),
    ],
    memory=sklearn_memory,
)

best_model_path = save_fitted_model(
    best_model_pipeline,
    f"best_optuna_{best_trial_summary['Modell']}",
    X_train,
    y_train,
)

print("Gespeichertes finales Optuna-Modell:", best_model_path)

Bestes Modell: Linear SVC unweighted
Trial: 6
Macro-F1: 0.8575
Generalization Gap Macro-F1: 0.1289


,Modell,trial_number,objective_value,penalty,alpha,l1_ratio,C,max_iter,tol,average,train_accuracy_mean,train_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,train_macro_f1_mean,train_macro_f1_std,train_weighted_f1_mean,train_weighted_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,fit_time_total_seconds
0,Linear SVC unweighted,6,0.8575,NaN,NaN,NaN,0.425808,30000,0.0005,NaN,0.9931,0.0002,0.9829,0.0012,0.9864,0.0007,0.9931,0.0002,0.9425,0.0011,0.8415,0.0211,0.8575,0.0142,0.9418,0.0011,16.13


,Wert
Modell,Linear SVC unweighted
trial_number,6
objective_value,0.8575
penalty,NaN
alpha,NaN
l1_ratio,NaN
C,0.425808
max_iter,30000
tol,0.0005
average,NaN


Gespeichertes finales Optuna-Modell: d:\toydev\schwarz-test\models\optuna\best_optuna_linear_svc_unweighted.joblib


# 7. Hyperparameteroptimierung der neuen Modelle

Nach den klassischen scikit-learn-Modellen werden auch die beiden erweiterten NLP-Modelle `fastText` und BERT für eine Hyperparameteroptimierung vorbereitet. Diese Modelle verwenden keine `ColumnTransformer`-Pipeline. Stattdessen werden die verwendeten Spalten zuerst zu einem gemeinsamen Textfeld zusammengeführt.

Die Optimierung erfolgt weiterhin nur auf den Trainingsdaten. Zur Bewertung wird innerhalb der Trainingsdaten ein stratifizierter Validierungssplit verwendet. Der finale Testdatensatz bleibt unverändert unberührt.


### 7.1. Gemeinsame Textbasis für fastText und BERT

fastText und BERT erwarten jeweils einen zusammenhängenden Text pro Beobachtung. Deshalb werden die zuvor ausgewählten Text- und kategorialen Spalten zu einem markierten Text zusammengeführt. Die Feldnamen bleiben erhalten, damit Informationen wie `name`, `geber`, `zweck`, `art` und `jahr` nicht vollständig vermischt werden.


In [25]:
import time
import unicodedata

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


nlp_model_output_dir = model_output_dir / "new_nlp_models"
nlp_model_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


nlp_feature_columns = feature_columns_optuna.copy()

X_train_nlp = X_train.copy()
X_train_nlp["model_text"] = combine_columns_as_text(
    X_train_nlp,
    nlp_feature_columns,
)

nlp_train_texts, nlp_valid_texts, nlp_y_train, nlp_y_valid = train_test_split(
    X_train_nlp["model_text"],
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train,
)

print("Verwendete Spalten für neue NLP-Modelle:")
display(
    pd.DataFrame(
        {"Trainingsspalte": nlp_feature_columns}
    )
)

print("Trainingssplit für BERT-Holdout:", len(nlp_y_train))
print("Validierungssplit für BERT-Holdout:", len(nlp_y_valid))

display(
    X_train_nlp[["model_text"]].head(3)
)


Verwendete Spalten für neue NLP-Modelle:


,Trainingsspalte
0,name_standardised
1,geber_standardised
2,anschrift_standardised
3,zweck_standardised
4,art_standardised
5,jahr


Trainingssplit für BERT-Holdout: 41328
Validierungssplit für BERT-Holdout: 4592


,model_text
id,
33655,"name: cashmere radio e. v. geber: senatsverwaltung für kultur und gesellschaftlicher zusammenhalt anschrift: frankfurter allee 7, 10247 berlin-bezirk friedrichshain-kreuzberg, deutschland zweck: signal2noise ? art, aesthetics and social practice of community radios art: projektförderung jahr: 2024"
105910,"name: merantix labs gmbh geber: senatsverwaltung für wirtschaft, energie und betriebe anschrift: c/o ai campus, fachbereich bionik und evolutionstechnik, max-urich-strasse 3, 13355 berlin, deutschland zweck: errichtung einer betriebsstätte art: projektförderung jahr: 2022"
67465,"name: georg kolbe-stiftung geber: senatsverwaltung für kultur und europa anschrift: sensburger allee 25, 14055 berlin-bezirk charlottenburg-wilmersdorf, deutschland zweck: der absolute tanz- festival sculpture art: projektförderung jahr: 2020"


### 7.2. fastText mit Optuna optimieren

fastText ist schnell genug, um mehrere Hyperparameterkombinationen auszuprobieren. Optimiert werden unter anderem Lernrate, Anzahl der Epochen, Wort-n-Gramme, Zeichen-n-Gramme und die Einbettungsdimension. Als Zielwert wird wie zuvor der Macro-F1-Score auf dem internen Validierungssplit verwendet.


In [26]:
fasttext_output_dir = nlp_model_output_dir / "fasttext"
fasttext_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


fasttext_label_mapping = (
    pd.DataFrame(
        {"politikbereich": sorted(y_train.unique())}
    )
    .assign(
        fasttext_label=lambda dataframe: dataframe["politikbereich"].map(
            make_fasttext_label
        )
    )
)

if fasttext_label_mapping["fasttext_label"].duplicated().any():
    raise ValueError(
        "Nicht eindeutige fastText-Labels gefunden. "
        "Bitte Label-Erzeugung prüfen."
    )

fasttext_label_to_class = dict(
    zip(
        fasttext_label_mapping["fasttext_label"],
        fasttext_label_mapping["politikbereich"],
    )
)

fasttext_label_mapping.to_csv(
    fasttext_output_dir / "fasttext_label_mapping.csv",
    index=False,
    sep=";",
    encoding="utf-8",
)


def write_fasttext_file(
    path,
    texts,
    labels,
):
    lines = (
        labels.map(make_fasttext_label)
        + " "
        + texts
        .fillna("")
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
    )

    path.write_text(
        "\n".join(lines.tolist()),
        encoding="utf-8",
    )


fasttext_full_train_path = fasttext_output_dir / "full_train.txt"

write_fasttext_file(
    fasttext_full_train_path,
    X_train_nlp["model_text"],
    y_train,
)

try:
    import fasttext

    def evaluate_fasttext_model(
        model,
        texts,
        y_true,
    ):
        predicted_labels, _ = model.predict(
            texts.tolist(),
            k=1,
        )

        y_pred = pd.Series(
            [
                fasttext_label_to_class.get(label_group[0], label_group[0])
                if len(label_group) > 0
                else None
                for label_group in predicted_labels
            ],
            index=y_true.index,
        )

        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true,
                y_pred,
            ),
        }


    def objective_fasttext(trial):
        params = {
            "epoch": trial.suggest_int("epoch", 5, 40),
            "lr": trial.suggest_float("lr", 0.05, 1.0, log=True),
            "wordNgrams": trial.suggest_int("wordNgrams", 1, 3),
            "dim": trial.suggest_categorical("dim", [50, 100, 200]),
            "minn": trial.suggest_categorical("minn", [0, 2, 3]),
            "loss": trial.suggest_categorical("loss", ["softmax", "ova"]),
        }

        if params["minn"] == 0:
            params["maxn"] = 0
        else:
            params["maxn"] = trial.suggest_int("maxn", 4, 7)

        fold_metric_rows = []
        total_training_time = 0.0

        for fold_number, (train_index, valid_index) in enumerate(
            cv.split(X_train_nlp["model_text"], y_train),
            start=1,
        ):
            fold_train_texts = X_train_nlp["model_text"].iloc[train_index]
            fold_valid_texts = X_train_nlp["model_text"].iloc[valid_index]
            fold_y_train = y_train.iloc[train_index]
            fold_y_valid = y_train.iloc[valid_index]

            fold_train_path = (
                fasttext_output_dir
                / f"trial_{trial.number}_fold_{fold_number}_train.txt"
            )
            write_fasttext_file(
                fold_train_path,
                fold_train_texts,
                fold_y_train,
            )

            start_time = time.perf_counter()
            model = fasttext.train_supervised(
                input=str(fold_train_path),
                verbose=0,
                **params,
            )
            training_time = time.perf_counter() - start_time
            total_training_time += training_time

            metrics = evaluate_fasttext_model(
                model,
                fold_valid_texts,
                fold_y_valid,
            )
            metrics["fold"] = fold_number
            metrics["fit_time_seconds"] = training_time
            fold_metric_rows.append(metrics)

        fold_metrics = pd.DataFrame(fold_metric_rows)

        summary = {
            "CV_Folds": n_splits,
            "validation_strategy": "shared_stratified_4_fold_cv",
            "fit_time_total_seconds": round(total_training_time, 2),
        }

        for metric_name in [
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "balanced_accuracy",
        ]:
            summary[f"val_{metric_name}_mean"] = round(
                fold_metrics[metric_name].mean(),
                4,
            )
            summary[f"val_{metric_name}_std"] = round(
                fold_metrics[metric_name].std(ddof=0),
                4,
            )
            summary[f"{metric_name}_mean"] = summary[
                f"val_{metric_name}_mean"
            ]
            summary[f"{metric_name}_std"] = summary[
                f"val_{metric_name}_std"
            ]

        for metric_name, metric_value in summary.items():
            trial.set_user_attr(metric_name, metric_value)

        with mlflow.start_run(
            run_name=f"fastText trial {trial.number}",
            nested=True,
        ):
            mlflow.log_params(params)
            mlflow.log_param("model", "fastText")
            mlflow.log_param(
                "validation_strategy",
                "shared_stratified_4_fold_cv",
            )
            for metric_name, metric_value in summary.items():
                if isinstance(metric_value, (int, float)):
                    mlflow.log_metric(metric_name, metric_value)

        return summary[f"val_{main_metric}_mean"]


    fasttext_study = optuna.create_study(
        study_name="fasttext_text_categorical_macro_f1_cv4",
        direction="maximize",
        storage=optuna_storage_url,
        load_if_exists=True,
        sampler=sampler,
    )

    n_trials_fasttext = 25

    fasttext_study.optimize(
        objective_fasttext,
        n_trials=n_trials_fasttext,
        show_progress_bar=True,
    )

    fasttext_tuning_results = build_result_table(
        fasttext_study,
        "fastText",
    )

    fasttext_tuning_results_path = (
        optuna_results_dir
        / "optuna_fasttext_results.csv"
    )
    fasttext_tuning_results.to_csv(
        fasttext_tuning_results_path,
        index=False,
        sep=";",
        encoding="utf-8",
    )

    best_fasttext_params = fasttext_study.best_trial.params.copy()
    if best_fasttext_params.get("minn", 0) == 0:
        best_fasttext_params["maxn"] = 0

    best_fasttext_model = fasttext.train_supervised(
        input=str(fasttext_full_train_path),
        verbose=0,
        **best_fasttext_params,
    )

    best_fasttext_model_path = (
        fasttext_output_dir
        / "best_fasttext_politikbereich.bin"
    )
    best_fasttext_model.save_model(str(best_fasttext_model_path))

    print("Bestes fastText-Modell gespeichert:", best_fasttext_model_path)
    display(fasttext_tuning_results.head(10))

except ImportError:
    fasttext_tuning_results = pd.DataFrame()
    print(
        "fastText ist nicht installiert. "
        "Installiere bei Bedarf fasttext und führe diese Zelle erneut aus."
    )


[I 2026-08-03 21:50:47,778] A new study created in RDB with name: fasttext_text_categorical_macro_f1_cv4


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-08-03 21:52:50,452] Trial 0 finished with value: 0.625 and parameters: {'epoch': 25, 'lr': 0.21865041067729277, 'wordNgrams': 1, 'dim': 50, 'minn': 3, 'loss': 'softmax', 'maxn': 5}. Best is trial 0 with value: 0.625.
[I 2026-08-03 21:53:30,061] Trial 1 finished with value: 0.5301 and parameters: {'epoch': 5, 'lr': 0.806752462672169, 'wordNgrams': 2, 'dim': 50, 'minn': 3, 'loss': 'softmax', 'maxn': 6}. Best is trial 0 with value: 0.625.
[I 2026-08-03 22:03:01,978] Trial 2 finished with value: 0.737 and parameters: {'epoch': 38, 'lr': 0.4022774121838481, 'wordNgrams': 2, 'dim': 200, 'minn': 3, 'loss': 'softmax', 'maxn': 6}. Best is trial 2 with value: 0.737.
[I 2026-08-03 22:08:06,607] Trial 3 finished with value: 0.3231 and parameters: {'epoch': 17, 'lr': 0.12048744643529845, 'wordNgrams': 3, 'dim': 200, 'minn': 3, 'loss': 'ova', 'maxn': 7}. Best is trial 2 with value: 0.737.
[I 2026-08-03 22:13:21,871] Trial 4 finished with value: 0.5809 and parameters: {'epoch': 37, 'lr': 0.13

,Modell,trial_number,objective_value,epoch,lr,wordNgrams,dim,minn,loss,maxn,CV_Folds,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,fit_time_total_seconds,macro_f1_mean,macro_f1_std,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std,validation_strategy,weighted_f1_mean,weighted_f1_std
0,fastText,15,0.8247,18,0.709317,1,100,0,softmax,NaN,4,0.9338,0.0027,0.8063,0.0216,42.34,0.8247,0.0151,0.9338,0.0027,0.8063,0.0216,0.8247,0.0151,0.9332,0.0028,shared_stratified_4_fold_cv,0.9332,0.0028
1,fastText,23,0.8224,16,0.866294,1,100,0,softmax,NaN,4,0.9338,0.0030,0.8049,0.0237,34.39,0.8224,0.0182,0.9338,0.0030,0.8049,0.0237,0.8224,0.0182,0.9333,0.0032,shared_stratified_4_fold_cv,0.9333,0.0032
2,fastText,21,0.8207,21,0.760544,2,100,0,softmax,NaN,4,0.9357,0.0032,0.8021,0.0294,52.53,0.8207,0.0247,0.9357,0.0032,0.8021,0.0294,0.8207,0.0247,0.9351,0.0033,shared_stratified_4_fold_cv,0.9351,0.0033
3,fastText,16,0.8152,18,0.876527,1,100,0,softmax,NaN,4,0.9334,0.0030,0.8014,0.0258,42.33,0.8152,0.0185,0.9334,0.0030,0.8014,0.0258,0.8152,0.0185,0.9327,0.0031,shared_stratified_4_fold_cv,0.9327,0.0031
4,fastText,22,0.7962,20,0.790188,3,100,0,softmax,NaN,4,0.9353,0.0025,0.7791,0.0183,54.99,0.7962,0.0135,0.9353,0.0025,0.7791,0.0183,0.7962,0.0135,0.9347,0.0026,shared_stratified_4_fold_cv,0.9347,0.0026
5,fastText,19,0.7948,19,0.658832,2,100,0,softmax,NaN,4,0.9358,0.0030,0.7773,0.0221,49.21,0.7948,0.0168,0.9358,0.0030,0.7773,0.0221,0.7948,0.0168,0.9352,0.0031,shared_stratified_4_fold_cv,0.9352,0.0031
6,fastText,24,0.7890,15,0.473983,1,100,0,softmax,NaN,4,0.9326,0.0033,0.7709,0.0203,32.31,0.7890,0.0154,0.9326,0.0033,0.7709,0.0203,0.7890,0.0154,0.9320,0.0035,shared_stratified_4_fold_cv,0.9320,0.0035
7,fastText,14,0.7861,19,0.300695,1,100,0,softmax,NaN,4,0.9321,0.0028,0.7666,0.0158,44.68,0.7861,0.0133,0.9321,0.0028,0.7666,0.0158,0.7861,0.0133,0.9314,0.0029,shared_stratified_4_fold_cv,0.9314,0.0029
8,fastText,17,0.7859,14,0.938723,3,100,0,softmax,NaN,4,0.9352,0.0027,0.7658,0.0142,42.70,0.7859,0.0141,0.9352,0.0027,0.7658,0.0142,0.7859,0.0141,0.9346,0.0028,shared_stratified_4_fold_cv,0.9346,0.0028
9,fastText,12,0.7837,30,0.182746,1,100,0,softmax,NaN,4,0.9320,0.0032,0.7638,0.0157,66.40,0.7837,0.0123,0.9320,0.0032,0.7638,0.0157,0.7837,0.0123,0.9313,0.0034,shared_stratified_4_fold_cv,0.9313,0.0034


### 7.3. BERT mit Optuna vorbereiten

BERT kann ebenfalls optimiert werden, ist aber deutlich rechenintensiver. Deshalb ist die BERT-Optimierung standardmäßig deaktiviert. In Colab mit GPU kann `run_bert_tuning = True` gesetzt werden. Optimiert werden nur wenige zentrale Parameter: Lernrate, Batch Size, Weight Decay, maximale Sequenzlänge und Anzahl der Epochen.


In [27]:
# bert_output_dir = nlp_model_output_dir / "bert"
# bert_output_dir.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# run_bert_tuning = False
# n_trials_bert = 5
# bert_model_checkpoint = "deepset/gbert-base"

# bert_label_encoder = LabelEncoder()
# bert_label_encoder.fit(y_train)

# bert_label_mapping = pd.DataFrame(
#     {
#         "label_id": range(len(bert_label_encoder.classes_)),
#         "politikbereich": bert_label_encoder.classes_,
#     }
# )
# bert_label_mapping.to_csv(
#     bert_output_dir / "bert_label_mapping.csv",
#     index=False,
#     sep=";",
#     encoding="utf-8",
# )
# dump(
#     bert_label_encoder,
#     bert_output_dir / "bert_label_encoder.joblib",
# )

# bert_y_train = bert_label_encoder.transform(nlp_y_train)
# bert_y_valid = bert_label_encoder.transform(nlp_y_valid)


# if run_bert_tuning:
#     try:
#         import shutil
#         import torch
#         from transformers import (
#             BertForSequenceClassification,
#             BertTokenizer,
#             Trainer,
#             TrainingArguments,
#         )

#         class EncodedPolitikbereichDataset(torch.utils.data.Dataset):
#             """Vor-tokenisierter Datensatz für BERT."""

#             def __init__(
#                 self,
#                 encodings,
#                 labels,
#             ):
#                 self.encodings = encodings
#                 self.labels = labels

#             def __len__(self):
#                 return len(self.labels)

#             def __getitem__(self, index):
#                 item = {
#                     key: torch.tensor(value[index])
#                     for key, value in self.encodings.items()
#                 }
#                 item["labels"] = torch.tensor(
#                     self.labels[index],
#                     dtype=torch.long,
#                 )
#                 return item

#         def make_training_arguments(
#             output_dir,
#             max_epochs,
#             learning_rate,
#             batch_size,
#             weight_decay,
#         ):
#             common_kwargs = {
#                 "output_dir": str(output_dir),
#                 "num_train_epochs": max_epochs,
#                 "learning_rate": learning_rate,
#                 "per_device_train_batch_size": batch_size,
#                 "per_device_eval_batch_size": max(batch_size, 8),
#                 "weight_decay": weight_decay,
#                 "logging_steps": 100,
#                 "save_strategy": "epoch",
#                 "load_best_model_at_end": True,
#                 "metric_for_best_model": "macro_f1",
#                 "greater_is_better": True,
#                 "report_to": "none",
#             }

#             try:
#                 return TrainingArguments(
#                     eval_strategy="epoch",
#                     **common_kwargs,
#                 )
#             except TypeError:
#                 return TrainingArguments(
#                     evaluation_strategy="epoch",
#                     **common_kwargs,
#                 )

#         def compute_bert_metrics(eval_prediction):
#             logits, labels = eval_prediction
#             predictions = np.argmax(logits, axis=-1)

#             return {
#                 "val_accuracy": accuracy_score(labels, predictions),
#                 "val_macro_f1": f1_score(
#                     labels,
#                     predictions,
#                     average="macro",
#                     zero_division=0,
#                 ),
#                 "val_weighted_f1": f1_score(
#                     labels,
#                     predictions,
#                     average="weighted",
#                     zero_division=0,
#                 ),
#                 "val_balanced_accuracy": balanced_accuracy_score(
#                     labels,
#                     predictions,
#                 ),
#             }

#         bert_tokenizer = BertTokenizer.from_pretrained(
#             bert_model_checkpoint,
#             token=hf_token,
#         )

#         def objective_bert(trial):
#             learning_rate = trial.suggest_float(
#                 "learning_rate",
#                 1e-5,
#                 5e-5,
#                 log=True,
#             )
#             batch_size = trial.suggest_categorical(
#                 "batch_size",
#                 [8, 16],
#             )
#             weight_decay = trial.suggest_float(
#                 "weight_decay",
#                 0.0,
#                 0.1,
#             )
#             max_length = trial.suggest_categorical(
#                 "max_length",
#                 [128, 256],
#             )
#             num_train_epochs = trial.suggest_categorical(
#                 "num_train_epochs",
#                 [1, 2, 3],
#             )

#             train_encodings = bert_tokenizer(
#                 nlp_train_texts.tolist(),
#                 truncation=True,
#                 padding="max_length",
#                 max_length=max_length,
#             )
#             valid_encodings = bert_tokenizer(
#                 nlp_valid_texts.tolist(),
#                 truncation=True,
#                 padding="max_length",
#                 max_length=max_length,
#             )

#             train_dataset = EncodedPolitikbereichDataset(
#                 train_encodings,
#                 bert_y_train,
#             )
#             valid_dataset = EncodedPolitikbereichDataset(
#                 valid_encodings,
#                 bert_y_valid,
#             )

#             model = BertForSequenceClassification.from_pretrained(
#                 bert_model_checkpoint,
#                 token=hf_token,
#                 num_labels=len(bert_label_encoder.classes_),
#                 id2label={
#                     index: label
#                     for index, label in enumerate(bert_label_encoder.classes_)
#                 },
#                 label2id={
#                     label: index
#                     for index, label in enumerate(bert_label_encoder.classes_)
#                 },
#             )

#             trial_output_dir = bert_output_dir / f"trial_{trial.number}"
#             training_args = make_training_arguments(
#                 trial_output_dir,
#                 num_train_epochs,
#                 learning_rate,
#                 batch_size,
#                 weight_decay,
#             )

#             trainer = Trainer(
#                 model=model,
#                 args=training_args,
#                 train_dataset=train_dataset,
#                 eval_dataset=valid_dataset,
#                 compute_metrics=compute_bert_metrics,
#             )

#             start_time = time.perf_counter()
#             trainer.train()
#             eval_metrics = trainer.evaluate()
#             training_time = time.perf_counter() - start_time

#             metrics = {
#                 key.replace("eval_", ""): value
#                 for key, value in eval_metrics.items()
#                 if key.startswith("eval_")
#             }
#             metrics["training_time_seconds"] = round(training_time, 2)

#             for metric_name, metric_value in metrics.items():
#                 trial.set_user_attr(metric_name, metric_value)

#             with mlflow.start_run(
#                 run_name=f"BERT trial {trial.number}",
#                 nested=True,
#             ):
#                 mlflow.log_param("model", bert_model_checkpoint)
#                 mlflow.log_param("learning_rate", learning_rate)
#                 mlflow.log_param("batch_size", batch_size)
#                 mlflow.log_param("weight_decay", weight_decay)
#                 mlflow.log_param("max_length", max_length)
#                 mlflow.log_param("num_train_epochs", num_train_epochs)
#                 for metric_name, metric_value in metrics.items():
#                     if isinstance(metric_value, (int, float)):
#                         mlflow.log_metric(metric_name, metric_value)

#             shutil.rmtree(
#                 trial_output_dir,
#                 ignore_errors=True,
#             )

#             return metrics[main_metric]

#         bert_study = optuna.create_study(
#             study_name="bert_text_categorical_macro_f1",
#             direction="maximize",
#             storage=optuna_storage_url,
#             load_if_exists=True,
#             sampler=sampler,
#         )

#         bert_study.optimize(
#             objective_bert,
#             n_trials=n_trials_bert,
#             show_progress_bar=True,
#         )

#         bert_tuning_results = build_result_table(
#             bert_study,
#             "German BERT",
#         )

#         bert_tuning_results_path = (
#             optuna_results_dir
#             / "optuna_bert_results.csv"
#         )
#         bert_tuning_results.to_csv(
#             bert_tuning_results_path,
#             index=False,
#             sep=";",
#             encoding="utf-8",
#         )

#         display(bert_tuning_results.head(10))

#     except ImportError:
#         bert_tuning_results = pd.DataFrame()
#         print(
#             "transformers und/oder torch sind nicht installiert. "
#             "Installiere die Pakete und führe diese Zelle erneut aus."
#         )
# else:
#     bert_tuning_results = pd.DataFrame()
#     print(
#         "BERT-Tuning ist deaktiviert. "
#         "Setze run_bert_tuning = True, um es in einer GPU-Umgebung auszuführen."
#     )


### 7.4. Ergebnisse der neuen Modelle zusammenführen

Die Ergebnisse der neuen NLP-Modelle werden separat gespeichert. Dadurch bleiben die klassischen Optuna-Ergebnisse reproduzierbar, während fastText und BERT als erweiterte Vergleichsmodelle ausgewertet werden können.


In [28]:
extended_result_frames = []

if "fasttext_tuning_results" in globals() and not fasttext_tuning_results.empty:
    extended_result_frames.append(fasttext_tuning_results)

# if "bert_tuning_results" in globals() and not bert_tuning_results.empty:
#     extended_result_frames.append(bert_tuning_results)

if extended_result_frames:
    extended_nlp_tuning_results = (
        pd.concat(
            extended_result_frames,
            ignore_index=True,
        )
        .sort_values(
            "objective_value",
            ascending=False,
        )
        .reset_index(drop=True)
    )
else:
    extended_nlp_tuning_results = pd.DataFrame()

extended_nlp_results_path = (
    optuna_results_dir
    / "optuna_extended_nlp_results.csv"
)

extended_nlp_tuning_results.to_csv(
    extended_nlp_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

print("Gespeichert:", extended_nlp_results_path)

if not extended_nlp_tuning_results.empty:
    display_model_evaluation_table(
        extended_nlp_tuning_results,
        model_column_candidates=[
            "Modell",
            "model",
        ],
        include_columns=[
            "trial_number",
            "objective_value",
            "training_time_seconds",
            "fit_time_total_seconds",
        ],
        metric_prefixes=[
            "train_",
            "val_",
        ],
        sort_by="objective_value",
        ascending=False,
    )
else:
    display(extended_nlp_tuning_results)


Gespeichert: d:\toydev\schwarz-test\data\processed\optuna_extended_nlp_results.csv


,Modell,trial_number,objective_value,fit_time_total_seconds,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_weighted_f1_mean,val_weighted_f1_std
0,fastText,15,0.8247,42.34,0.9338,0.0027,0.8063,0.0216,0.8247,0.0151,0.9332,0.0028
1,fastText,23,0.8224,34.39,0.9338,0.0030,0.8049,0.0237,0.8224,0.0182,0.9333,0.0032
2,fastText,21,0.8207,52.53,0.9357,0.0032,0.8021,0.0294,0.8207,0.0247,0.9351,0.0033
3,fastText,16,0.8152,42.33,0.9334,0.0030,0.8014,0.0258,0.8152,0.0185,0.9327,0.0031
4,fastText,22,0.7962,54.99,0.9353,0.0025,0.7791,0.0183,0.7962,0.0135,0.9347,0.0026
5,fastText,19,0.7948,49.21,0.9358,0.0030,0.7773,0.0221,0.7948,0.0168,0.9352,0.0031
6,fastText,24,0.7890,32.31,0.9326,0.0033,0.7709,0.0203,0.7890,0.0154,0.9320,0.0035
7,fastText,14,0.7861,44.68,0.9321,0.0028,0.7666,0.0158,0.7861,0.0133,0.9314,0.0029
8,fastText,17,0.7859,42.70,0.9352,0.0027,0.7658,0.0142,0.7859,0.0141,0.9346,0.0028
9,fastText,12,0.7837,66.40,0.9320,0.0032,0.7638,0.0157,0.7837,0.0123,0.9313,0.0034


Die erweiterten NLP-Ergebnisse werden separat zusammengeführt, weil fastText und BERT nicht dieselbe scikit-learn-Pipeline verwenden. fastText kann mit begrenzter Rechenzeit relativ schnell getestet werden. BERT bleibt dagegen optional, da ein systematisches Fine-Tuning deutlich höhere GPU-Ressourcen erfordert.
